# IS480 2022 World Cup Project Phase 3

## Here you'll find:
1. Verification of imported warehouse tables
2. Analytical queries in Spark SQL
3. Advanced aggregations (ROLLUP, CUBE, RANK)
4. PySpark-based Analysis
5. Cross-platform Comparison/Scalability Discussion


## 1. Verify the uploaded tables exist

These are the tables created from the CSV uploads:
- `dim_date`
- `dim_match`
- `dim_stage`
- `dim_team`
- `dim_tournament`
- `fact_match_team_performance`


In [0]:
spark.sql("SELECT COUNT(*) AS dim_team_rows FROM dim_team").show()
spark.sql("SELECT COUNT(*) AS dim_match_rows FROM dim_match").show()
spark.sql("SELECT COUNT(*) AS dim_date_rows FROM dim_date").show()
spark.sql("SELECT COUNT(*) AS dim_stage_rows FROM dim_stage").show()
spark.sql("SELECT COUNT(*) AS dim_tournament_rows FROM dim_tournament").show()
spark.sql("SELECT COUNT(*) AS fact_rows FROM fact_match_team_performance").show()

+-------------+
|dim_team_rows|
+-------------+
|           32|
+-------------+

+--------------+
|dim_match_rows|
+--------------+
|             8|
+--------------+

+-------------+
|dim_date_rows|
+-------------+
|            7|
+-------------+

+--------------+
|dim_stage_rows|
+--------------+
|             6|
+--------------+

+-------------------+
|dim_tournament_rows|
+-------------------+
|                  1|
+-------------------+

+---------+
|fact_rows|
+---------+
|       15|
+---------+



## 2. Total Goals by Team

This query calculates total goals for each team. It confirms that the fact and dimension tables join correctly and that aggregation works in Spark.


In [0]:
spark.sql("""
SELECT
    d.team_name,
    SUM(f.goals) AS total_goals
FROM fact_match_team_performance f
JOIN dim_team d
    ON f.team_key = d.team_key
GROUP BY d.team_name
ORDER BY total_goals DESC, d.team_name
""").display()

team_name,total_goals
France,7
England,6
Argentina,4
Ecuador,2
Iran,2
Japan,2
Netherlands,2
Saudi Arabia,2
Australia,1
Spain,1


Databricks visualization. Run in Databricks to view.

## 3. Ranking Teams by Goals

This query ranks teams based on total goals using a window function. It shows how ranking can be applied to compare performance.


In [0]:
spark.sql("""
SELECT
    team_name,
    total_goals,
    RANK() OVER (ORDER BY total_goals DESC) AS team_rank
FROM (
    SELECT
        d.team_name,
        SUM(f.goals) AS total_goals
    FROM fact_match_team_performance f
    JOIN dim_team d
        ON f.team_key = d.team_key
    GROUP BY d.team_name
) ranked_teams
ORDER BY team_rank, team_name
""").display()

team_name,total_goals,team_rank
France,7,1
England,6,2
Argentina,4,3
Ecuador,2,4
Iran,2,4
Japan,2,4
Netherlands,2,4
Saudi Arabia,2,4
Australia,1,9
Spain,1,9


Databricks visualization. Run in Databricks to view.

## 4. Goals by Stage (ROLLUP)

This query aggregates goals by tournament stage and includes a total using ROLLUP. It shows how scoring changes across stages.

In [0]:
spark.sql("""
SELECT
    COALESCE(s.stage_name, 'ALL STAGES') AS stage_name,
    SUM(f.goals) AS total_goals
FROM fact_match_team_performance f
JOIN dim_stage s
    ON f.stage_key = s.stage_key
GROUP BY ROLLUP(s.stage_name)
ORDER BY stage_name
""").display()

stage_name,total_goals
ALL STAGES,29
Final,6
Group Stage,23
Round of 16,0


Databricks visualization. Run in Databricks to view.

## 5. Performance by Team and Stage (CUBE)

This query uses CUBE to analyze goals across both team and stage. It generates multiple aggregation combinations for deeper analysis.

In [0]:
spark.sql("""
SELECT
    COALESCE(d.team_name, 'ALL TEAMS') AS team_name,
    COALESCE(s.stage_name, 'ALL STAGES') AS stage_name,
    SUM(f.goals) AS total_goals
FROM fact_match_team_performance f
JOIN dim_team d
    ON f.team_key = d.team_key
JOIN dim_stage s
    ON f.stage_key = s.stage_key
GROUP BY CUBE(d.team_name, s.stage_name)
ORDER BY team_name, stage_name
""").show(100, truncate=False)

+------------+-----------+-----------+
|team_name   |stage_name |total_goals|
+------------+-----------+-----------+
|ALL TEAMS   |ALL STAGES |29         |
|ALL TEAMS   |Final      |6          |
|ALL TEAMS   |Group Stage|23         |
|ALL TEAMS   |Round of 16|0          |
|Argentina   |ALL STAGES |4          |
|Argentina   |Final      |3          |
|Argentina   |Group Stage|1          |
|Argentina   |Round of 16|0          |
|Australia   |ALL STAGES |1          |
|Australia   |Group Stage|1          |
|Ecuador     |ALL STAGES |2          |
|Ecuador     |Group Stage|2          |
|England     |ALL STAGES |6          |
|England     |Group Stage|6          |
|France      |ALL STAGES |7          |
|France      |Final      |3          |
|France      |Group Stage|4          |
|Iran        |ALL STAGES |2          |
|Iran        |Group Stage|2          |
|Japan       |ALL STAGES |2          |
|Japan       |Group Stage|2          |
|Netherlands |ALL STAGES |2          |
|Netherlands |Group Stage

## 6. Team Playstyle Classification

Description:

This query classifies teams into playstyles using CASE logic based on performance metrics. It shows how raw stats can be turned into categories.


In [0]:
spark.sql("""
WITH team_style AS (
    SELECT
        d.team_name,
        CASE
            WHEN AVG(f.possession_pct) >= 55 AND AVG(f.passes_completed / NULLIF(f.passes, 0)) >= 0.80
                THEN 'possession-dominant'
            WHEN AVG(f.possession_pct) < 45 AND AVG(f.total_attempts) >= 8
                THEN 'counter-attack'
            ELSE 'balanced'
        END AS playstyle,
        SUM(CASE WHEN f.goals > f.goals_conceded THEN 1 ELSE 0 END) AS wins
    FROM fact_match_team_performance f
    JOIN dim_team d
        ON f.team_key = d.team_key
    GROUP BY d.team_name
)
SELECT
    playstyle,
    COUNT(*) AS team_count,
    SUM(wins) AS total_wins
FROM team_style
GROUP BY playstyle
ORDER BY total_wins DESC, team_count DESC
""").show()

+-------------------+----------+----------+
|          playstyle|team_count|total_wins|
+-------------------+----------+----------+
|           balanced|        10|         5|
|possession-dominant|         1|         1|
+-------------------+----------+----------+



## 7. Matchup Difficulty (PySpark)

This section builds a custom difficulty score using PySpark. It combines multiple stats to estimate how strong each team is.

In [0]:
from pyspark.sql import functions as F

fact_df = spark.table("fact_match_team_performance")
team_df = spark.table("dim_team").select("team_key", "team_name")

matchup_difficulty_df = (
    fact_df.groupBy("team_key")
    .agg(
        F.avg("goals").alias("avg_goals"),
        F.avg("total_attempts").alias("avg_attempts"),
        F.avg("defensive_pressures_applied").alias("avg_pressure"),
        (
            F.sum("passes_completed") / F.when(F.sum("passes") == 0, None).otherwise(F.sum("passes"))
        ).alias("pass_completion_rate")
    )
    .fillna(0)
    .withColumn(
        "matchup_difficulty_score",
        F.round(
            F.col("avg_goals") * 0.4 +
            F.col("avg_attempts") * 0.2 +
            F.col("avg_pressure") * 0.2 +
            F.col("pass_completion_rate") * 0.2,
            3
        )
    )
    .join(team_df, on="team_key", how="left")
    .select(
        "team_name",
        "avg_goals",
        "avg_attempts",
        "avg_pressure",
        "pass_completion_rate",
        "matchup_difficulty_score"
    )
    .orderBy(F.desc("matchup_difficulty_score"))
)

matchup_difficulty_df.show(truncate=False)

+------------+------------------+------------+------------+--------------------+------------------------+
|team_name   |avg_goals         |avg_attempts|avg_pressure|pass_completion_rate|matchup_difficulty_score|
+------------+------------------+------------+------------+--------------------+------------------------+
|England     |6.0               |0.0         |0.0         |0.9130434782608695  |2.583                   |
|France      |3.5               |0.0         |0.0         |0.0                 |1.4                     |
|Iran        |2.0               |0.0         |0.0         |0.8476190476190476  |0.97                    |
|Japan       |2.0               |0.0         |0.0         |0.0                 |0.8                     |
|Saudi Arabia|2.0               |0.0         |0.0         |0.0                 |0.8                     |
|Netherlands |2.0               |0.0         |0.0         |0.0                 |0.8                     |
|Ecuador     |2.0               |0.0         |

## 8. Cross-platform Comparison 
In this section, we compare how the same queries were written and executed in Oracle SQL, Spark SQL, and PySpark. Oracle and Spark SQL are very similar in syntax, but Spark runs on a distributed system which makes it better for handling larger datasets. PySpark takes a different approach by using DataFrames instead of writing raw SQL, which makes it more flexible for building more complex logic like our matchup difficulty analysis.

- Oracle SQL → traditional relational database queries
- Spark SQL → similar syntax but runs on distributed engine
- PySpark → uses DataFrame API instead of pure SQL
- Spark scales better for large datasets

## 9. Scalability Discussion
As data size increases, traditional databases like Oracle can struggle because they typically run on a single machine. Spark, on the other hand, is designed for distributed processing, meaning it can split large datasets across multiple nodes and process them in parallel. This allows Spark to handle much larger datasets (such as billions of rows) more efficiently than a single-node system.